# 03b - Pipeline en Cascade
## Objectif : Prédire les variables contemporaines puis le CA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import joblib
import warnings

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
print("✅ Imports OK")

✅ Imports OK


In [2]:
df_ml = pd.read_csv('/app/notebooks/ml_dataset_corr.csv', parse_dates=['date'])
TARGET = 'total_sales'

print(f"✅ {len(df_ml)} mois | {df_ml['date'].min().strftime('%b %Y')} → {df_ml['date'].max().strftime('%b %Y')}")
df_ml.head(3)

✅ 37 mois | Jan 2023 → Jan 2026


,year,month,quarter,total_sales,total_quantity,nb_documents,nb_clients,nb_produits,nb_regions,nb_warehouses,...,total_cost,total_marge,date,semester,is_summer,is_end_year,is_january,lag_1,lag_12,rolling_mean_3
0,2023,1,1,"1,037,923.46",16598,1380,223,4963,9,4,...,"1,011,052.67","26,870.79",2023-01-01,1,0,0,1,"985,659.74","591,407.94","960,005.64"
1,2023,2,1,"939,044.09",15537,1280,227,4505,9,4,...,"939,085.12",-41.03,2023-02-01,1,0,0,0,"1,037,923.46","576,097.45","993,566.44"
2,2023,3,1,"948,325.79",15961,1392,240,4353,9,4,...,"932,696.74","15,629.04",2023-03-01,1,0,0,0,"939,044.09","680,213.89","987,542.43"


## 1. Définir les variables à prédire en cascade

In [3]:
# Variables contemporaines qu'on va prédire en premier
TARGETS_CASCADE = ['nb_clients', 'nb_documents', 'total_quantity']

# Features de base — uniquement temporelles + lags
BASE_FEATURES = [
    'month', 'quarter', 'semester',
    'is_summer', 'is_end_year', 'is_january',
    'lag_1', 'lag_2', 'lag_3',
    'rolling_mean_3', 'rolling_mean_6', 'lag_12'
]

# Vérifier que toutes les colonnes existent
print("BASE_FEATURES disponibles :")
for f in BASE_FEATURES:
    ok = '✅' if f in df_ml.columns else '❌'
    print(f"  {ok} {f}")

BASE_FEATURES disponibles :
  ✅ month
  ✅ quarter
  ✅ semester
  ✅ is_summer
  ✅ is_end_year
  ✅ is_january
  ✅ lag_1
  ❌ lag_2
  ❌ lag_3
  ✅ rolling_mean_3
  ❌ rolling_mean_6
  ✅ lag_12


## 2. Split temporel

In [4]:
split_idx  = int(len(df_ml) * 0.85)
split_date = df_ml.iloc[split_idx]['date']

df_train = df_ml.iloc[:split_idx].copy()
df_test  = df_ml.iloc[split_idx:].copy()

print(f"Train : {len(df_train)} mois ({df_train['date'].min().strftime('%b %Y')} → {df_train['date'].max().strftime('%b %Y')})")
print(f"Test  : {len(df_test)} mois ({df_test['date'].min().strftime('%b %Y')} → {df_test['date'].max().strftime('%b %Y')})")

Train : 31 mois (Jan 2023 → Jul 2025)
Test  : 6 mois (Aug 2025 → Jan 2026)


## 3. Entraîner un modèle pour chaque variable contemporaine

In [5]:
cascade_models  = {}
cascade_scalers = {}
cascade_metrics = {}

for target_var in TARGETS_CASCADE:
    print(f"\n{'='*45}")
    print(f"🎯 Modèle pour : {target_var}")
    print(f"{'='*45}")

    X_tr = df_train[BASE_FEATURES]
    y_tr = df_train[target_var]
    X_te = df_test[BASE_FEATURES]
    y_te = df_test[target_var]

    sc       = StandardScaler()
    X_tr_sc  = sc.fit_transform(X_tr)
    X_te_sc  = sc.transform(X_te)

    m = Ridge(alpha=10)
    m.fit(X_tr_sc, y_tr)
    pred = m.predict(X_te_sc)

    mae  = mean_absolute_error(y_te, pred)
    mape = np.mean(np.abs((y_te - pred) / y_te)) * 100
    r2   = r2_score(y_te, pred)

    cascade_models[target_var]  = m
    cascade_scalers[target_var] = sc
    cascade_metrics[target_var] = {'MAE': mae, 'MAPE': mape, 'R²': r2}

    print(f"MAE  : {mae:>12,.2f}")
    print(f"MAPE : {mape:>12.2f} %")
    print(f"R²   : {r2:>12.4f}")

print(f"\n✅ {len(cascade_models)} modèles intermédiaires entraînés")


🎯 Modèle pour : nb_clients


KeyError: "['lag_2', 'lag_3', 'rolling_mean_6'] not in index"